# Epsilon Fund — Cross-Sectional Momentum (Walk-Forward)

At each rebalance, ranks every coin by trailing J-day return, goes long the top N
and short the bottom N (equal weight, dollar-neutral), holds for K days, then re-ranks.

Uses `infrastructure/walkforward/` for rolling Optuna optimisation and OOS evaluation.
Data pulled via `topics/momentum/xs_momentum/xs_data.py`.

---
### Parameters
| Parameter | Description | Range |
|---|---|---|
| `J` | Formation / lookback window (days) | 7–30 |
| `K` | Holding period (days) | 7–14 |
| `top_n` | Coins per leg (long + short simultaneously) | 2–4 |

3 free parameters — within the heuristic budget for a 730-bar training window.

---
### Iteration guidelines

| Signal | Meaning | Action |
|--------|---------|--------|
| IS Sharpe drops, OOS Sharpe holds or rises | Removing noise-fitting degrees of freedom | Continue iterating |
| Perturbation degradation shrinks across iterations | Parameters becoming more robust | Continue iterating |
| WFE improvement flattens | Diminishing returns | Stop iterating |
| OOS Sharpe keeps declining despite fixes | Overfitting the iteration process | Stop — strategy architecture problem |


In [32]:
import sys
import os
import pandas as pd
import numpy as np

# ── repo root — works on both Mac and Windows ────────────────────────────────
ROOT = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = os.path.join('C:\\', 'Users', 'user', 'Documents', 'Epsilon Fund', 'Epsilon-Quant-Research')
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'backtester'))
sys.path.insert(0, os.path.join(ROOT, 'topics', 'momentum', 'xs_momentum'))

from binance_client  import get_binance_client
from wf_engine       import (walk_forward, plateau_analysis, plateau_summary,
                              perturbation_test, cost_stress_test)
from wf_visualizer   import plot_walk_forward_results, plot_plateau_analysis
from ls_diagnostics  import compute_attribution, plot_attribution
from engine          import backtest
from xs_data         import load_xs_data, build_close_panel, DEFAULT_COINS
from universe_filter import load_cache, get_universe, precompute_avg_volume

---
## Data

Two modes, controlled by `USE_DYNAMIC_UNIVERSE` in the next cell:

**Static mode (`False`)** — fast iteration with a hardcoded coin list (`DEFAULT_COINS`).
Pulls daily OHLCV directly from Binance via `load_xs_data()`. Good for quick parameter
sweeps where the universe is fixed.

**Dynamic mode (`True`)** — universe is re-selected at each rebalance from a local
parquet cache built by `universe_cache.py`. The cache holds **USDT perpetual futures
data only** — perp prices feed both the ranking signal and the backtest PnL (matching
what would actually trade on Hyperliquid for the dollar-neutral L/S strategy). At each
rebalance, `get_universe(dates[r-1], ...)` filters by 30-day avg perp volume and listing
age — strictly no lookahead since only data up to the signal formation date is used.

**Cache maintenance:** run `python universe_cache.py` once to build the cache, then
once per day to incrementally append yesterday's candle. Notebook runs make zero
API calls — they read parquet directly.

**`COST`** — per-leg trading cost fraction. The strategy returns gross
`strategy_returns` plus a `turnover` column; the engine applies
`trade_cost = turnover × COST` automatically. Adjust `COST` without re-running
the strategy — `cost_stress_test()` rescales it for free.

In [44]:
# ══════════════════════════════════════════════════════════════════════════════
# MODE
# ══════════════════════════════════════════════════════════════════════════════
USE_DYNAMIC_UNIVERSE = True   # True  → load from perp cache (universe_cache.py)
                               # False → fetch API with COINS list below

# ══════════════════════════════════════════════════════════════════════════════
# UNIVERSE FILTER CONFIG  (used when USE_DYNAMIC_UNIVERSE = True)
# ══════════════════════════════════════════════════════════════════════════════
UF_TOP_N          = 20          # max coins in the eligible universe at each rebalance
UF_MIN_VOLUME     = 200_000_000  # min 30-day avg daily USDT perp volume ($)
UF_MIN_AGE_DAYS   = 365         # min days since listing (perp onboard date)
UF_VOLUME_WINDOW  = 30          # rolling window for avg volume computation

# ══════════════════════════════════════════════════════════════════════════════
# STATIC CONFIG  (used when USE_DYNAMIC_UNIVERSE = False)
# ══════════════════════════════════════════════════════════════════════════════
COINS    = DEFAULT_COINS   # ['BTC','ETH','SOL','BNB','XRP','DOGE','ADA','AVAX','LINK','MATIC']
INTERVAL = '1d'
LOOKBACK = 2200            # days; covers ~6 years for most coins

# ══════════════════════════════════════════════════════════════════════════════
# SHARED
# ══════════════════════════════════════════════════════════════════════════════
COST = 0.001   # per-leg trading cost fraction

# ── Static data load (API, only when USE_DYNAMIC_UNIVERSE = False) ────────────
if not USE_DYNAMIC_UNIVERSE:
    client = get_binance_client()
    print('Fetching daily OHLCV...')
    data = load_xs_data(coins=COINS, interval=INTERVAL, lookback=LOOKBACK, client=client)

    panel    = build_close_panel(data)
    volume   = None    # signals no dynamic filtering inside the strategy
    meta     = None

    ew_equity = (1 + panel.pct_change().mean(axis=1).fillna(0)).cumprod()
    ew_df     = pd.DataFrame({'Close': ew_equity}, index=panel.index)
    driver_df = data['BTC'][['Close']].copy()

    print(f'\nClose panel: {panel.shape}  ({panel.index[0].date()} → {panel.index[-1].date()})')
    print(f'Driver (BTC): {len(driver_df)} bars')

In [45]:
if USE_DYNAMIC_UNIVERSE:
    # ── Load from parquet cache (no API calls) ────────────────────────────────
    # Run  python universe_cache.py  first to build / update the cache.
    print('Dynamic universe mode — loading perp cache...')
    close_full, volume, meta = load_cache()

    print(f'  close  : {close_full.shape}')
    print(f'  volume : {volume.shape}')
    print(f'  meta   : {meta.shape}')

    # ── Clip price panel to LOOKBACK days from the most recent cached date ───
    # `volume` and `meta` intentionally KEEP full history so get_universe's
    # rolling 30-day avg has enough bars even when as_of_date sits at the
    # start of the LOOKBACK window.  Slicing by `as_of_date <= dates[r-1]`
    # inside get_universe still prevents lookahead.
    cutoff     = close_full.index[-1] - pd.Timedelta(days=LOOKBACK)
    close_full = close_full.loc[close_full.index >= cutoff]

    # ── Restrict panel to coins with a recorded listing date in meta ─────────
    tradeable = sorted([c for c in close_full.columns
                         if c in meta.index and pd.notna(meta.loc[c, 'listing_date'])])
    panel = close_full[tradeable]

    # ── Equal-weight basket benchmark ────────────────────────────────────────
    ew_equity = (1 + panel.pct_change().mean(axis=1).fillna(0)).cumprod()
    ew_df     = pd.DataFrame({'Close': ew_equity}, index=panel.index)

    # ── Driver (BTC perp — provides the walk-forward date index) ─────────────
    driver_df = panel[['BTC']].rename(columns={'BTC': 'Close'}).copy()

    print(f'\nTradeable universe: {len(tradeable)} coins')
    print(f'Date range: {panel.index[0].date()} → {panel.index[-1].date()}  '
          f'({len(panel)} bars, clipped to LOOKBACK={LOOKBACK})')
    print(f'volume kept full ({len(volume)} bars) for clean rolling-avg lookback')
else:
    print('Static mode — data already loaded in Cell 3.')

Dynamic universe mode — loading perp cache...
  close  : (2428, 526)
  volume : (2428, 526)
  meta   : (526, 1)

Tradeable universe: 526 coins
Date range: 2020-04-22 → 2026-05-01  (2201 bars, clipped to LOOKBACK=2200)
volume kept full (2428 bars) for clean rolling-avg lookback


---
## ⚠️  WF Engine Interface — Multi-Asset Adapter

### What the engine expects

`walk_forward()` was designed for single-asset strategies:
```
strategy_fn(df_slice, params)  →  (df, indicator_cols)
```
where `df_slice` is **one coin's** OHLCV for the training or test window, and the
returned `df` must carry a `position` column (`1`/`0`/`-1`). The engine slices a single
`df` by bar index to build train/test windows.

### Why XS momentum doesn't fit

1. **Multi-asset data** — `strategy_fn` receives only one coin's slice, but XS
   momentum needs all coins simultaneously for cross-sectional ranking.
2. **Portfolio returns, not a single position** — the output is a weighted long/short
   portfolio return; there is no single `1`/`0`/`-1` column that makes sense.

### The adapter used here (no engine modifications)

**Closure + `strategy_returns` path:**

- `make_xs_strategy(full_panel, cost_per_leg)` captures the full multi-asset close
  panel as a Python closure.
- Inside `strategy_fn`, `df_slice` is ignored **except for its `DatetimeIndex`**, which
  is used to window `full_panel` to the correct train/test dates.
- The strategy computes per-bar portfolio returns (including transaction costs) and
  stores them in a `strategy_returns` column.
- `position = 1` is set everywhere, activating the engine's precomputed-returns path:
  ```python
  # engine.py — precomputed path
  net_returns  = position.shift(1) * strategy_returns   # bar-0 zeroed (see below)
  equity_curve = (1 + net_returns).cumprod()
  ```
- `cost=0` is passed to `walk_forward` because costs are **already baked** into
  `strategy_returns`. Passing any non-zero cost would double-count.

### Three consequences to be aware of

**1. Bar-0 of every slice has zero return.**  
The engine applies `eff_pos = position.shift(1)`, so `eff_pos[0] = 0` regardless of
`position[0] = 1`. This zeros bar-0's return in every train and test window. On a
daily strategy with 700+ bars per fold the effect is negligible (~0.14% of bars).

**2. `num_trades = 0` — custom `reject_fn` is required.**  
The engine detects trades via `position.diff()`. With `position=1` always,
`position.diff() = 0` everywhere, so no trades are logged. The **default** `reject_fn`
has `if num_trades < 7: return True` which would reject every Optuna trial. The
custom `reject_fn` below uses Sharpe and drawdown filters instead.

**3. `cost_stress_test()` is replaced by a manual version.**  
The built-in `cost_stress_test()` re-runs the OOS DataFrame at higher costs by
re-calling `engine.backtest()`, but because `position=1` always the engine sees no
position changes and adds zero cost. The manual stress test re-runs `strategy_fn`
at escalating `cost_per_leg` values on the OOS date window instead.

### What a proper multi-asset engine would need

If the XS strategy family grows, the engine would need these targeted changes:

| Change | File | Where |
|--------|------|-------|
| Accept `data_dict` alongside `df` so folds slice all coins | `wf_engine.py` | `walk_forward()` signature |
| Pass the full per-coin slice dict to `strategy_fn` | `wf_engine.py` | `_make_objective()` and fold loop |
| Allow `strategy_fn` to return a position *matrix* (bars × coins) | `wf_engine.py` | return contract |
| Aggregate portfolio returns from the position matrix before calling `backtest()` | `wf_engine.py` | `_run_backtest()` |

The closure adapter keeps the engine untouched and is fully sufficient for a
single XS strategy. Consider the engine changes only if you need the WF framework
to also optimise per-coin position sizing or dynamic universe selection.

---

---
## Parameter Configuration

Define which parameters to optimise and which to anchor.

**First run:** leave `FIXED_PARAMS` empty — let Optuna search all three dimensions
freely. After the stability table prints, copy parameters with CV < 0.15 into
`FIXED_PARAMS` and re-run.

| Param | Range | Notes |
|---|---|---|
| `J` | 7–30 days | Standard daily momentum formation window (1 week to 1 month) |
| `K` | 7–14 days | Holding period; shorter = more rebalances = more costs |
| `top_n` | 2–4 coins | Integer; 3 means 3 long + 3 short out of 10 coins (30% each leg) |

In [ ]:
# ── search ranges ─────────────────────────────────────────────────────────────
# Format: 'param_name': ('int' | 'float', lo, hi)
#
# pct_long / pct_short = fraction of the eligible universe to put in each leg.
# Inside the strategy: n_long  = max(1, int(pct_long  * universe_size))   (floor)
#                     n_short = max(1, int(pct_short * universe_size))   (floor)
# Decile portfolios (10%) are the academic standard for XS momentum;
# 20% is a common variant. Beyond ~30% the ranking signal degrades fast.
PARAM_DEFS = {
    'J':         ('int',   7,    45),    # formation / lookback window (days)
    'K':         ('int',   7,    21),    # holding period (days)
    'pct_long':  ('float', 0.05, 0.30),  # fraction of universe in long leg
    'pct_short': ('float', 0.05, 0.30),  # fraction of universe in short leg
}

# ── fixed params ──────────────────────────────────────────────────────────────
# Start empty — anchor stable parameters (CV < 0.15) after the first stability run.
FIXED_PARAMS = {}

# Example after a stability run:
# FIXED_PARAMS = {
#     'pct_long':  0.20,
#     'pct_short': 0.20,
# }

---
## Strategy

**Logic:**
1. Compute trailing J-day return for every coin in the (dynamic) universe.
2. Rank coins by return (descending). Pick the top `pct_long` fraction as the long leg
   and the bottom `pct_short` fraction as the short leg.
3. Hold the equal-weight dollar-neutral portfolio for K bars, then re-rank.

**Why percentile, not absolute count:**
The universe size varies across rebalances (filter returns 8–30 coins depending on
liquidity / listing-age constraints at each date). Picking a fixed `n_long = 3` from
a 30-coin universe is "top decile" — strong tail signal. Picking `n_long = 3` from
a 6-coin universe is "top half" — mostly noise. Percentile-based selection keeps the
*nature* of the strategy consistent regardless of universe size.

**Per-rebalance count derivation:**
- `n_long  = max(1, floor(pct_long  × universe_size))`
- `n_short = max(1, floor(pct_short × universe_size))`
- If `n_long + n_short > universe_size` (very thin universe), the rebalance is skipped (flat for K bars).

**Signal lag:** `window.pct_change(J).shift(1)` — at bar `r`, the signal uses the
J-day return ending at bar `r-1`. Execution occurs at bar `r`. No lookahead bias.

**Cost model:** the strategy returns gross portfolio returns. A `turnover` column
(values 0–2) records the fractional portfolio turnover at each rebalance bar.
The engine applies `trade_cost = turnover × COST` so costs flow through Optuna,
OOS evaluation, and `cost_stress_test()` automatically.

Turnover is computed from actual position overlap across rebalances (uses the
*per-rebalance* `n_long` / `n_short` since universe size and selection counts vary):

  - `long_turn  = |new_long  − old_long|  / n_long_this_rebalance`
  - `short_turn = |new_short − old_short| / n_short_this_rebalance`
  - `turnover   = long_turn + short_turn`            ∈ [0, 2]

In [ ]:
def make_xs_strategy(panel, volume=None, meta=None,
                     uf_top_n=None, uf_min_volume=None, uf_min_age=None,
                     uf_volume_window=None):
    """
    Factory that captures the perp data via closure.
    Returns a strategy_fn compatible with wf_engine.walk_forward().

    Single perp panel for both signal and PnL — perp prices feed the ranking
    signal and the backtest execution.  This matches what would actually trade
    on Hyperliquid for the dollar-neutral L/S strategy.

    Performance
    -----------
    Two-level optimisation for fast Optuna runs:
      1. Rolling avg-volume panel pre-computed ONCE here (shared across all trials)
         → O(1) lookup inside get_universe instead of recomputing rolling mean.
      2. Inner bar loop fully vectorised via numpy:
         - perp_rets converted to a numpy matrix once per call
         - Long / short basket returns computed for the whole K-bar holding
           window at once (np.nanmean over coin axis), bulk-assigned to
           pre-allocated numpy arrays.
      Combined effect: ~10–20× speedup vs the naive implementation.

    Output columns:
      strategy_returns, turnover, long_ret, short_ret, universe_size

    Dynamic universe (no lookahead): when volume and meta are provided,
    get_universe(dates[r-1], ...) is called at every rebalance.

    Turnover:
      long_turn  = |new_long  − old_long|  / n_long    ∈ [0, 1]
      short_turn = |new_short − old_short| / n_short   ∈ [0, 1]
      turnover   = long_turn + short_turn               ∈ [0, 2]
    """
    # ── Resolve filter defaults once ──────────────────────────────────────────
    _top_n      = uf_top_n         if uf_top_n         is not None else len(panel.columns)
    _min_vol    = uf_min_volume    if uf_min_volume    is not None else 50_000_000
    _min_age    = uf_min_age       if uf_min_age       is not None else 180
    _vol_window = uf_volume_window if uf_volume_window is not None else 30

    # ── Pre-compute rolling avg volume ONCE (shared across all Optuna trials) ─
    avg_vol_panel = (precompute_avg_volume(volume, volume_window=_vol_window)
                     if volume is not None else None)

    # ── Pre-compute coin → column-index mapping ──────────────────────────────
    coin_to_idx = {c: i for i, c in enumerate(panel.columns)}

    def strategy_fn(df_slice, params):
        J         = int(params['J'])
        K         = int(params['K'])
        pct_long  = float(params['pct_long'])
        pct_short = float(params['pct_short'])

        # ── Window panel to this fold's date range ────────────────────────────
        dates = df_slice.index
        win   = panel.reindex(dates)
        n     = len(dates)

        mom_signal = win.pct_change(J).shift(1)        # 1-bar lag, no lookahead
        rets_arr   = win.pct_change().values            # numpy view for fast slicing

        # Pre-allocated output arrays
        strategy_returns = np.full(n, np.nan)
        long_returns     = np.full(n, np.nan)
        short_returns    = np.full(n, np.nan)
        turnover_arr     = np.zeros(n)
        universe_size    = np.zeros(n)

        prev_long  = []
        prev_short = []

        # ── Rebalance loop ────────────────────────────────────────────────────
        for r in range(J + 1, n, K):
            signal = mom_signal.iloc[r]

            # Dynamic universe: eligible coins at dates[r-1] (signal formation)
            if volume is not None and meta is not None:
                eligible = get_universe(
                    as_of_date     = dates[r - 1],
                    volume         = volume,
                    meta           = meta,
                    top_n          = _top_n,
                    min_avg_volume = _min_vol,
                    min_age_days   = _min_age,
                    volume_window  = _vol_window,
                    avg_vol_panel  = avg_vol_panel,
                )
                eligible = [c for c in eligible if c in coin_to_idx]
                if eligible:
                    signal = signal[eligible]

            valid = signal.dropna()

            # ── Percentile → absolute counts (floor, min 1) ───────────────
            n_long  = max(1, int(pct_long  * len(valid)))
            n_short = max(1, int(pct_short * len(valid)))

            if len(valid) < (n_long + n_short):
                continue   # universe too thin for disjoint long/short

            ranked      = valid.sort_values(ascending=False)
            long_coins  = ranked.index[:n_long].tolist()
            short_coins = ranked.index[-n_short:].tolist()

            universe_size[r] = len(valid)

            if prev_long and prev_short:
                long_turn  = len(set(long_coins)  - set(prev_long))  / n_long
                short_turn = len(set(short_coins) - set(prev_short)) / n_short
            else:
                long_turn  = 1.0
                short_turn = 1.0
            turnover_arr[r] = long_turn + short_turn

            # ── Vectorised holding-period PnL ─────────────────────────────────
            hold_end  = min(r + K, n)
            long_idx  = [coin_to_idx[c] for c in long_coins]
            short_idx = [coin_to_idx[c] for c in short_coins]

            long_slice  = rets_arr[r:hold_end][:, long_idx]
            short_slice = rets_arr[r:hold_end][:, short_idx]

            long_basket  = np.nanmean(long_slice,  axis=1)
            short_basket = np.nanmean(short_slice, axis=1)

            long_returns[r:hold_end]     = long_basket
            short_returns[r:hold_end]    = short_basket
            strategy_returns[r:hold_end] = 0.5 * long_basket - 0.5 * short_basket

            prev_long  = long_coins
            prev_short = short_coins

        # ── Convert numpy arrays back to pd.Series ───────────────────────────
        # fillna(0) on the return series treats skipped rebalances and warmup
        # bars as 'strategy flat' (no position → 0 return) instead of missing
        # data.  Without this, wf_engine.dropna(subset=['strategy_returns'])
        # deletes the NaN rows, creating K-bar date gaps inside folds wherever
        # the universe filter briefly returned < (n_long + n_short) coins.
        # Warmup bars get filled with 0 too but are removed by the test_start
        # trim in wf_engine, so the net effect on warmup handling is unchanged.
        strategy_returns = pd.Series(strategy_returns, index=dates).fillna(0.0)
        long_returns     = pd.Series(long_returns,     index=dates).fillna(0.0)
        short_returns    = pd.Series(short_returns,    index=dates).fillna(0.0)
        turnover         = pd.Series(turnover_arr,     index=dates)
        universe_size    = pd.Series(universe_size,    index=dates)

        # ── Build output DataFrame ───────────────────────────────────────────
        sr_filled = strategy_returns.fillna(0.0)
        equity    = (1.0 + sr_filled).cumprod()

        result = pd.DataFrame({
            'Open':             equity.shift(1).bfill(),
            'High':             equity,
            'Low':              equity,
            'Close':            equity,
            'Volume':           1.0,
            'position':         1,
            'position_size':    1.0,
            'strategy_returns': strategy_returns,
            'turnover':         turnover,
            'long_ret':         long_returns,
            'short_ret':        short_returns,
            'universe_size':    universe_size,
        }, index=dates)

        return result, ['strategy_returns']

    return strategy_fn


my_strategy = make_xs_strategy(
    panel            = panel,
    volume           = volume,
    meta             = meta,
    uf_top_n         = UF_TOP_N,
    uf_min_volume    = UF_MIN_VOLUME,
    uf_min_age       = UF_MIN_AGE_DAYS,
    uf_volume_window = UF_VOLUME_WINDOW,
)

---
## Run Walk-Forward

**Fold setup for daily data:**

| Parameter | Value | Rationale |
|---|---|---|
| `TRAIN_BARS` | 1050 | ~3 years; many rebalances per fold at K=7-21 |
| `TEST_BARS` | 285 | ~9.5 months; ~3.7:1 train:test ratio |
| `BURNIN_BARS` | 50 | **MUST be >= max(J) + 1**. Currently max J = 45 in PARAM_DEFS, so 46 is the minimum; 50 leaves a safety margin. |
| `N_TRIALS` | 300 | 4 free params (J, K, pct_long, pct_short) |
| `cost` | **COST** | Per-leg fraction applied via `turnover × COST` in the engine |

**Why BURNIN_BARS matters:** the strategy's rebalance loop starts at `r = J+1`, so bars `0..J` of every strategy_fn call produce NaN `strategy_returns`. `wf_engine` drops those NaN rows via `dropna(subset=['strategy_returns'])` and *then* trims to `test_start`. If `BURNIN_BARS < max_J + 1`, the warmup overflows the burnin region into the test window — those test bars get permanently dropped, creating date gaps at every fold boundary. The visualizer renders these gaps as flat segments / jumps in the OOS equity curve. Always keep `BURNIN_BARS >= max(J) + 1`.

**Cost flow:** `strategy_fn` returns gross `strategy_returns` plus a `turnover`
column. The engine computes `trade_cost = turnover × cost` at each rebalance bar.
Costs propagate correctly through Optuna trials, OOS evaluation, plateau/perturbation
analysis, and `cost_stress_test()` without any special handling.

**Custom `reject_fn`:** `num_trades = 0` for this strategy (position=1 everywhere,
so no position changes are detected). The default filter `num_trades < 7` would
reject all trials. Sharpe ≥ 0 and drawdown are used as quality gates instead.

In [48]:
# ── walk-forward windows ──────────────────────────────────────────────────────
TRAIN_BARS  = 1050   # ~3 years daily
TEST_BARS   = 285    # ~9.5 months
BURNIN_BARS = 50     # MUST be >= max(J) + 1 in PARAM_DEFS (currently J_max=45).
                     # The strategy_fn rebalance loop starts at r=J+1, so bars 0..J
                     # produce NaN strategy_returns.  Those bars are dropped by
                     # wf_engine via dropna(subset=['strategy_returns']).  If burnin
                     # is too small, the dropped warmup overflows into the test
                     # window, creating date gaps at fold boundaries — visible as
                     # flat segments / jumps in the OOS equity curve.
N_TRIALS    = 300

# ── SCORING FUNCTION ──────────────────────────────────────────────────────────
def score_fn(m):
    SHARPE_MAX = 3.0
    CALMAR_MAX = 6.0
    RETURN_MAX = 10.0

    calmar = (m['total_return'] / abs(m['max_drawdown'])
              if m['max_drawdown'] != 0 else 0.0)

    s = np.clip(m['sharpe_ratio']  / SHARPE_MAX, 0, 1)
    c = np.clip(calmar             / CALMAR_MAX, 0, 1)
    r = np.clip(m['total_return']  / RETURN_MAX, 0, 1)

    return 0.60 * s + 0.25 * c + 0.15 * r


# ── REJECTION CRITERIA ────────────────────────────────────────────────────────
# Do NOT check num_trades — always 0 for this strategy (position=1 everywhere).

def reject_fn(m):
    if m is None:                   return True
    if m['sharpe_ratio'] < 0.0:     return True
    if m['max_drawdown'] < -0.80:   return True
    if m['total_return'] < 0.0:     return True
    return False


results = walk_forward(
    df           = driver_df,
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    train_bars   = TRAIN_BARS,
    test_bars    = TEST_BARS,
    burnin_bars  = BURNIN_BARS,
    n_trials     = N_TRIALS,
    cost         = COST,       # applied by engine as: trade_cost = turnover * COST
    score_fn     = score_fn,
    reject_fn    = reject_fn,
    save_csv     = None,
)


UPDATED WALK_FORWARD FILE IS RUNNING
Walk-forward: 4 fold(s)  train=1050  test=285  burnin=50  trials=300
  Fold 1: train 2020-04-22 → 2023-03-07  | test 2023-03-08 → 2023-12-17
  Fold 2: train 2021-02-01 → 2023-12-17  | test 2023-12-18 → 2024-09-27
  Fold 3: train 2021-11-13 → 2024-09-27  | test 2024-09-28 → 2025-07-09
  Fold 4: train 2022-08-25 → 2025-07-09  | test 2025-07-10 → 2026-04-20

────────────────────────────────────────────────────────────
Fold 1/4  train: 2020-04-22 → 2023-03-07  test: 2023-03-08 → 2023-12-17


  0%|          | 0/300 [00:00<?, ?it/s]


  IS  → Sharpe: 1.47  Return: 471.86%  DD: -45.66%  Calmar: 1.83  Trades: 0
  OOS → Sharpe: 0.06  Return: -3.88%  DD: -39.21%  Calmar: -0.13  Trades: 0

  Best params: {'J': 25, 'K': 17, 'n_long': 2, 'n_short': 6}

────────────────────────────────────────────────────────────
Fold 2/4  train: 2021-02-01 → 2023-12-17  test: 2023-12-18 → 2024-09-27


  0%|          | 0/300 [00:00<?, ?it/s]


  IS  → Sharpe: 1.30  Return: 124.17%  DD: -13.25%  Calmar: 2.44  Trades: 0
  OOS → Sharpe: -0.55  Return: -13.63%  DD: -23.20%  Calmar: -0.74  Trades: 0

  Best params: {'J': 23, 'K': 15, 'n_long': 9, 'n_short': 3}

────────────────────────────────────────────────────────────
Fold 3/4  train: 2021-11-13 → 2024-09-27  test: 2024-09-28 → 2025-07-09


  0%|          | 0/300 [00:00<?, ?it/s]


  IS  → Sharpe: 1.11  Return: 96.55%  DD: -14.95%  Calmar: 1.77  Trades: 0
  OOS → Sharpe: 1.54  Return: 35.59%  DD: -21.92%  Calmar: 2.18  Trades: 0

  Best params: {'J': 10, 'K': 9, 'n_long': 5, 'n_short': 7}

────────────────────────────────────────────────────────────
Fold 4/4  train: 2022-08-25 → 2025-07-09  test: 2025-07-10 → 2026-04-20


  0%|          | 0/300 [00:00<?, ?it/s]


  IS  → Sharpe: 1.45  Return: 332.57%  DD: -46.01%  Calmar: 1.44  Trades: 0
  OOS → Sharpe: -1.37  Return: -68.94%  DD: -76.66%  Calmar: -1.01  Trades: 0

  Best params: {'J': 7, 'K': 18, 'n_long': 2, 'n_short': 3}

════════════════════════════════════════════════════════════
WALK-FORWARD SUMMARY
════════════════════════════════════════════════════════════

Out-of-sample across 4 fold(s):
  Avg Sharpe:       -0.08
  Avg Return:       -12.7%
  Avg Max Drawdown: -40.2%
  Avg Calmar:       0.07
  Avg Trades/fold:  0
  Folds profitable: 1/4
  Sharpe OOS/IS:    -0.06  (poor)

────────────────────────────────────────────────────────────
COMBINED OOS: 2023-03-08 → 2026-04-20  (1140 bars)
  Return:        -63.85%
  Sharpe:        -0.39
  Max Drawdown:  -76.66%
  Calmar:        -0.36
  Profit Factor: 0.00
  Win Rate:      0.00%
  Num Trades:    0


---
## Granular Results and Parameter Stability

Per-fold IS vs OOS performance. Compare `train_*` vs `test_*` to assess overfitting.

| Column | Description |
|---|---|
| `*_sharpe` `*_return` `*_drawdown` `*_calmar` | Core performance metrics |
| `optuna_score` | Best score achieved on training window |
| `param_J` `param_K` `param_pct_long` `param_pct_short` | Best parameter values per fold |

**Note:** `test_trades`, `test_winrate`, `test_profit_factor` will all be 0 — the
engine logs no trades because `position = 1` everywhere (see Cell 4). Use Sharpe,
return, drawdown, **and the long/short attribution table further below** as the
primary OOS quality signals for this strategy.

**Consensus Parameters** — anchor parameters with CV < 0.15 in `FIXED_PARAMS`.
For percentile-based XS momentum, `pct_long` and `pct_short` are often the most
stable since the strategy is now scale-invariant to universe size.

In [ ]:
# ── fold summary table ────────────────────────────────────────────────────────
display_cols = [
    'train_start', 'train_end',
    'test_start',  'test_end',
    'train_sharpe', 'test_sharpe',
    'train_return', 'test_return',
    'train_drawdown', 'test_drawdown',
    'optuna_score',
    'param_J', 'param_K', 'param_pct_long', 'param_pct_short',
]
display(results['results_df'][display_cols].round(3))

# ── parameter stability ───────────────────────────────────────────────────────
stability = results['stability_df'].copy()
stability['stable'] = stability['stable'].map({True: 'yes', False: ''})
stability['fixed']  = stability['fixed'].map({True: 'yes', False: ''})
stability = stability[['param', 'median', 'std', 'cv', 'stable', 'fixed']].round(3)
print('\nParameter stability (CV < 0.15 = stable candidate for fixing):')
display(stability.sort_values('cv'))

# ── consensus params ──────────────────────────────────────────────────────────
stable = results['stability_df'][results['stability_df']['cv'] < 0.15]

print('Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:')
for _, row in stable.iterrows():
    v = results['consensus_params'][row['param']]
    v_fmt = (int(round(v)) if isinstance(v, float) and v == int(v)
             else round(v, 4) if isinstance(v, float) else v)
    print(f"    '{row['param']}': {v_fmt},")

print('\nConsensus parameters (median across folds):')
for k, v in results['consensus_params'].items():
    print(f'  {k:<12} = {round(v, 4) if isinstance(v, float) else v}')

---
## Parameter Robustness Checks

### Plateau Analysis
Sweeps each free parameter across its full range (holding others at consensus),
evaluating the score at each point over the full date range.

| Verdict | Plateau % | Meaning |
|---|---|---|
| Robust | ≥ 60% | Score stays near peak across most of the range |
| Moderate | 30–60% | Some sensitivity — watch if it degrades further after fixing |
| FRAGILE | < 30% | Strategy depends critically on hitting the exact value — consider anchoring |
| N/A | — | Every sweep point failed reject_fn — strategy completely intolerant on this axis |

### Perturbation Test
Randomly jitters **all** free parameters simultaneously by ±5/10/20% of their range.
Tests whether the optimum is a broad hill or a narrow spike in the joint parameter space.

> With only 3 integer parameters, this test is less informative than for 15-param
> strategies. Focus on the plateau % instead.

In [50]:
# ── 1-D sensitivity sweeps around consensus params ─────────────────────────
sweep_results = plateau_analysis(
    df           = driver_df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = 0.0,         # costs baked into strategy_returns
    reject_fn    = reject_fn,   # required: default reject_fn checks num_trades < 7 which is always 0
    n_steps      = 20,
)

# ── text verdicts ──────────────────────────────────────────────────────────
verdict_df = plateau_summary(
    sweep_results,
    base_params  = results['consensus_params'],
    stability_df = results['stability_df'],
    threshold    = 0.20,
)

# ── neighbourhood perturbation ─────────────────────────────────────────────
# With only 3 integer parameters the perturbation may hit boundary values often.
# Treat degradation > 20% at ±10% offset as a fragility signal.
perturb_df = perturbation_test(
    df           = driver_df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = 0.0,
    reject_fn    = reject_fn,   # required: default reject_fn checks num_trades < 7 which is always 0
    pct_offsets  = (0.05, 0.10, 0.20),
    n_samples    = 50,
)


═══════════════════════════════════════════════════════════════════════════════════════════
PLATEAU ANALYSIS — PARAMETER ROBUSTNESS
═══════════════════════════════════════════════════════════════════════════════════════════
Parameter                 Consensus Peak Score  Plateau %  Fold CV Verdict                 
──────────────────────── ────────── ────────── ────────── ──────── ────────────────────────
n_short                           4      0.114       40.0%    0.397 Moderate                
n_long                            4      0.176       22.2%    0.821 FRAGILE                 
K                                16      0.266       14.3%    0.218 FRAGILE                 
J                                16      0.267       11.1%    0.476 FRAGILE                 

═══════════════════════════════════════════════════════════════════════════
PERTURBATION TEST — NEIGHBOURHOOD ROBUSTNESS
═══════════════════════════════════════════════════════════════════════════
Base score: 0.0732
  

In [51]:

# ── plateau sweep charts ──────────────────────────────────────────────────────
plot_plateau_analysis(
    sweep_results    = sweep_results,
    consensus_params = results['consensus_params'],
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    threshold        = 0.20,
    show             = False,
    save_html        = None,
)


---
## Results Charts and Cost Stress Test

**OOS equity chart** — combined OOS period stitched from all folds, overlaid with the
equal-weight universe basket (benchmark).

**Why the equal-weight basket is the right benchmark for this strategy:**  
A dollar-neutral XS momentum portfolio (long top-N, short bottom-N, equal weight) has
**zero net market exposure** by construction — the long and short legs cancel in aggregate.
This means BTC buy-and-hold is *not* a valid comparison: if the whole market rises uniformly,
a dollar-neutral strategy should earn approximately zero from that move.

The equal-weight basket represents the passive return from holding all universe coins equally,
with no selection or ranking. Any outperformance over this baseline is *pure cross-sectional
alpha* — return that comes solely from the ranking signal correctly identifying relative
winners and losers, not from riding overall market direction.

Reading the chart:
- **Benchmark rises, strategy lags** → the short leg is the drag; you are short coins that
  are rising, just slower than the longs. The strategy is still doing its job, but market
  beta inside the long leg does not offset the short-leg drag.
- **Benchmark falls, strategy holds or rises** → the short leg is working; the coins you
  are short are falling harder than the longs, generating positive spread return.
- **Both fall together** → a broad market crash overwhelms cross-sectional differentiation;
  expected for XS momentum in high-correlation bear regimes where all assets move together.

**Cost stress test** — re-runs the combined OOS backtest at escalating cost multipliers.
Works correctly because `oos_combined_df` carries the `turnover` column: the engine applies
`trade_cost = turnover × cost` at each rebalance bar, so raising `cost` correctly scales
the burden on high-turnover periods without re-running the strategy.

In [52]:
# ── OOS equity + fold charts ──────────────────────────────────────────────────
plot_walk_forward_results(
    results         = results,
    param_defs      = PARAM_DEFS,
    fixed_params    = FIXED_PARAMS,
    benchmark_data  = ew_df,       # equal-weight universe basket
    show            = True,
    save_html_dir   = None,
    show_fold_perf  = False,
    show_param_evol = False,
    show_oos_equity = True,
    show_trades     = False,
)

# ── transaction cost stress test ──────────────────────────────────────────────
if results['oos_combined_df'] is not None:
    cost_df = cost_stress_test(
        oos_combined_df  = results['oos_combined_df'],
        cost_multipliers = (1.0, 1.5, 2.0, 3.0),
        base_cost        = COST,
    )
else:
    print('No combined OOS dataframe — skip cost stress test.')


═══════════════════════════════════════════════════════════════════════════
TRANSACTION COST STRESS TEST
═══════════════════════════════════════════════════════════════════════════
    Cost   Mult   Sharpe     Return      MaxDD   Calmar       PF
──────── ────── ──────── ────────── ────────── ──────── ────────
  0.0010   1.0x    -0.39    -63.85%    -76.66%    -0.36     0.00
  0.0015   1.5x    -0.42    -65.69%    -76.93%    -0.38     0.00
  0.0020   2.0x    -0.46    -67.44%    -77.20%    -0.39     0.00
  0.0030   3.0x    -0.52    -70.68%    -77.73%    -0.42     0.00


---
## Long/Short Attribution Diagnostics

Cross-sectional-specific diagnostics that replace trade-centric metrics
(meaningless here since `position=1` always — the engine logs zero trades).

**Attribution table** — Sharpe and total return decomposed into:
- `long_sharpe` / `long_total_return` — performance of being 100% long the long basket
- `short_sharpe` / `short_basket_total` — performance of being 100% short the short basket
- `spread_sharpe` / `spread_total` — dollar-neutral spread (long − short, 1× leverage)
- `net_sharpe` — combined post-cost return (matches the OOS Sharpe in the fold table)
- `hit_rate` — fraction of rebalance periods where (long − short) > 0
- `mean_turnover` — average fractional turnover at rebalance bars (range 0–2)
- `avg/min/max_universe_size` — # coins available at rebalance over the OOS period

**Single most important read:** if `short_sharpe < 0` consistently across runs, the
short leg is a drag rather than a profit centre. That argues for dropping it
(long-only) or re-thinking the ranking (the bottom of the universe might not be
mean-reverting losers — it might be coins that already crashed and bounce).

**4-panel chart** —
1. Cumulative equity by leg (long, −short, combined net) — visual long/short attribution
2. Cumulative spread (long − short) — the raw momentum premium harvested
3. Turnover per rebalance — high turnover = unstable rankings = noisy signal
4. Universe size over time — flags degraded regimes where ranking is meaningless

In [ ]:
# ── long/short attribution table + diagnostic chart ──────────────────────────
oos = results['oos_combined_df']

if oos is None or 'long_ret' not in oos.columns:
    print('No OOS dataframe with attribution columns — skip diagnostics.')
else:
    attr = compute_attribution(oos, ann=365)

    print('═' * 60)
    print('LONG/SHORT ATTRIBUTION')
    print('═' * 60)
    for k, v in attr.items():
        if isinstance(v, float):
            if 'sharpe' in k:
                print(f'  {k:<22} {v:>8.3f}')
            elif 'return' in k or 'total' in k or 'rate' in k or 'turnover' in k:
                print(f'  {k:<22} {v:>8.2%}' if abs(v) < 100 else f'  {k:<22} {v:>8.3f}')
            else:
                print(f'  {k:<22} {v:>8.2f}')
        else:
            print(f'  {k:<22} {v:>8}')

    plot_attribution(oos,
                     benchmark_data=ew_df,   # equal-weight basket → strips beta from leg returns
                     show=True, save_html=None,
                     title='XS Momentum — Long/Short Diagnostics (OOS)')

---
## Live Execution Notes (forward-looking — for when this goes live)

**Venue:** Binance (single account, no multi-venue complexity).

**Recommended structure:** spot-long + perp-short under cross-margin.
- **Long leg** → Binance spot (just hold the coin)
- **Short leg** → Binance USDT-M perp (open short position)
- Cross-margin lets the spot holdings collateralize the perp shorts → capital efficiency stays close to perp/perp.

**Why this beats perp/perp:**
On XS momentum specifically, the long leg picks the *highest-momentum* coins, which systematically have the *highest* funding rates (long-biased flow). The short leg picks the *lowest-momentum* coins, which usually have low / negative funding rates. So perp/perp pays a lot on longs and only partially offsets via shorts. Spot-long avoids the long-side cost entirely; perp-short still receives any short-side funding. Net funding moves from "drag" to "small positive."

**Funding-rate cache deferred.** With this structure funding is no longer a first-order cost — modeling it precisely is a refinement, not a gate. Build it later only if you also want to backtest a perp/perp variant for comparison. Architecture sketch (when needed):
- `infrastructure/data/funding_cache.py` — mirror `universe_cache.py` (incremental fetch, parquet output)
- `infrastructure/perps/funding.py` — `apply_funding(positions, funding_panel)` helper
- Sign convention: `funding_pnl = -position × funding_rate` (long pays when rate > 0)

**Backtest data note:** the perp cache built by `universe_cache.py` is fine as a price proxy for spot in the backtest. For top-30 liquid Binance coins, spot ↔ perp basis runs ~5–15 bps intraday, well under the noise floor of a daily strategy. Don't build a parallel spot cache just for backtesting purposes.

**Bigger missing items to model before going live (in priority order):**
1. **Slippage on the bottom of the universe.** Rank 25–30 by volume can be thinly traded — assume 5–15 bps per leg vs the close.
2. **Borrow availability** if you ever go spot-margin-short instead of perp-short (some coins are hard-to-borrow; rates spike in stressed regimes).
3. **Funding** — only after the above two are calibrated. For spot-long + perp-short it's a small positive contribution, not a drag.

In [54]:
# ── save OOS dataframe ────────────────────────────────────────────────────────
# Load in a portfolio notebook with:
#   xs_oos = pd.read_pickle(os.path.join('oos', 'xs_momentum_oos.pkl'))
if results['oos_combined_df'] is not None:
    oos_dir  = os.path.join(os.path.dirname(os.path.abspath('xs_momentum_wf.ipynb')), 'oos')
    os.makedirs(oos_dir, exist_ok=True)
    oos_path = os.path.join(oos_dir, 'xs_momentum_oos.pkl')
    results['oos_combined_df'].to_pickle(oos_path)
    print(f'Saved OOS dataframe  ({len(results["oos_combined_df"])} bars)  →  {oos_path}')
else:
    print('No combined OOS dataframe to save.')

Saved OOS dataframe  (1140 bars)  →  /Users/justiniturregui/Desktop/epsilon/github/Epsilon-Quant-Research/topics/momentum/xs_momentum/oos/xs_momentum_oos.pkl
